# KM2 — CNN na FashionMNIST z własną biblioteką autodiff

Uczenie konwolucyjnej sieci neuronowej do klasyfikacji obrazów FashionMNIST przy użyciu
własnej biblioteki do automatycznego różniczkowania (`flex.jl`). Sieć po 3 epokach osiąga
ok. 87% dokładności na zbiorze testowym.

In [ ]:
include("flex.jl")
using MLDatasets

train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

function loader(data; batchsize::Int=1)
    x4dim = reshape(Float32.(data.features), 28, 28, 1, :)
    yhot  = onehotbatch(data.targets, 0:9)
    DataLoader((x4dim, yhot); batchsize, shuffle=true)
end

net = Chain(
  Conv((3, 3), 1 => 6, pad=1, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, pad=1, bias=false),
  MaxPool((2, 2)),
  flatten,
  Dense(784 => 84, relu),
  Dropout(0.4),
  Dense(84 => 10),
)

In [ ]:
function loss_and_accuracy(model, data)
    model.training[] = false
    X = Float32.(data.features)
    Y = data.targets
    n = size(X, 3)
    total_loss = 0.0
    correct    = 0
    for i in 1:n
        xi   = reshape(X[:, :, i], 28, 28, 1)
        yi   = one_hot(Y[i])
        yhat = model(xi)
        total_loss += logitcrossentropy(yhat, yi)
        correct    += (onecold(yhat, 0:9) == Y[i])
    end
    model.training[] = true
    (; loss = total_loss / n, acc = round(100 * correct / n; digits=2))
end

settings = (; eta = 1e-2, epochs = 3, batchsize = 10)
@show loss_and_accuracy(net, test_data);

In [ ]:
opt = GradientDescent(settings.eta)
for epoch in 1:settings.epochs
    @time for (x_batch, y_batch) in loader(train_data, batchsize=settings.batchsize)
        for i in 1:size(x_batch, 4)
            zerograd!(net.graph)
            net.x_node.data .= x_batch[:, :, :, i]
            net.y_node.data .= y_batch[:, i]
            forward!(net.graph)
            backward!(net.graph)
            accumulate!(opt, net.graph)
        end
        optimize!(opt, net.graph)
    end

    _, train_acc = loss_and_accuracy(net, train_data)
    _, test_acc  = loss_and_accuracy(net, test_data)
    println("epoch=$epoch  train_acc=$train_acc  test_acc=$test_acc")
end

In [ ]:
@show loss_and_accuracy(net, test_data);